# Aegis Defense AI: Fine-Tuning Llama 3.2 on Cybersecurity Benchmarks

This notebook fine-tunes **Llama 3.2 3B Instruct** using QLoRA on curated cybersecurity datasets from [JunfengGo/Dataset-for-cybersecurity](https://github.com/JunfengGo/Dataset-for-cybersecurity):
1. **Network Intrusion Detection (NIDS)**: NSL-KDD benchmark (DoS, Probe, R2L, U2R telemetry).
2. **Web Application Firewall (WAF)**: FWAF malicious queries (SQLi, XSS, Path Traversal, Command Injection).
3. **Defensive Alignment & Playbooks**: MITRE ATT&CK techniques and incident containment guidelines.

### Recommended Environment: Free Google Colab T4 GPU (`Runtime -> Change runtime type -> T4 GPU`)

In [ ]:
# 1. Install required packages
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
import json
import urllib.request
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./aegis-defense-lora"

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

In [ ]:
# 3. Fetch or construct defensive cybersecurity instruction dataset
# Sample instructions grounded in NSL-KDD and FWAF benchmarks
sample_sft_data = [
    {
        "instruction": "You are Aegis-Defense, an elite autonomous cybersecurity defensive intelligence agent. Provide actionable analysis for threat detection, incident triage, and hardening.",
        "input": "Security Analyst Prompt: Explain the threat profile, indicators of compromise, and defensive mitigation strategy for SYN Flood (Neptune) (T1498.001).",
        "output": "### Threat Profile: SYN Flood (Neptune)\n- **Category:** DoS\n- **MITRE ATT&CK Technique:** T1498.001\n- **Classification:** CWE-400\n- **Severity Level:** HIGH\n\n### Overview & Mechanism\nExhausts server transmission control block (TCB) state by transmitting thousands of TCP SYN packets without completing the three-way handshake.\n\n### Indicators of Compromise (IoCs)\nHigh count of inbound TCP SYN packets with flag S0/REJ, identical destination ports, 0 dst_bytes, elevated serror_rate (>0.90).\n\n### Defensive Mitigation Playbook\nEnable SYN Cookies (net.ipv4.tcp_syncookies = 1), lower synack retries, configure iptables rate-limiting, and utilize upstream cloud DDoS scrubbing."
    },
    {
        "instruction": "You are Aegis-Defense, an elite autonomous cybersecurity defensive intelligence agent. Analyze network connection telemetry and identify potential intrusions.",
        "input": "Analyze network telemetry flow: Protocol=tcp, Service=private, Flag=S0, SrcBytes=0, DstBytes=0, ConnectionCount=120, SerrorRate=1.00, RerrorRate=0.00.",
        "output": "### Traffic Analysis: Anomaly Detected [DOS]\n- **Attack Signature:** neptune (SYN Flood)\n- **Threat Class:** DoS\n- **Anomalous Indicators:** Flag `S0` with 100% SYN error rate across 120 connection bursts and 0 response bytes.\n- **Defensive Action:** Activate TCP SYN cookies, apply iptables rate-limiting for offending subnet, inspect state table, and alert SOC analysts."
    },
    {
        "instruction": "You are Aegis-Defense, an elite autonomous cybersecurity defensive intelligence agent. Inspect web queries for OWASP Top 10 vulnerabilities.",
        "input": "Inspect HTTP request parameter: /api/users?id=1' UNION SELECT null, username, password FROM users--",
        "output": "### WAF Inspection: Malicious Query Detected [CRITICAL]\n- **Attack Type:** SQL Injection (SQLi) - UNION-based exfiltration\n- **CWE:** CWE-89\n- **Recommended Action:** BLOCK\n- **Remediation:** Enforce parameterized queries (Prepared Statements/ORM). Deploy WAF regex rules blocking UNION SELECT queries in parameter values."
    }
]

formatted = []
for r in sample_sft_data:
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{r['instruction']}<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{r['input']}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n{r['output']}<|eot_id|>"
    formatted.append({"text": text})

dataset = Dataset.from_list(formatted)
print(f"Prepared {len(dataset)} training examples.")

In [ ]:
# 4. Load Tokenizer & Model with 4-bit Quantization
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Train with SFTTrainer
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Successfully saved LoRA adapter to {OUTPUT_DIR}")

In [ ]:
# 6. Test Fine-Tuned Model Inference
prompt = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are Aegis-Defense, an elite autonomous cybersecurity defensive intelligence agent.<|eot_id|><|start_header_id|>user<|end_header_id|>\nAnalyze network telemetry flow: Protocol=icmp, Service=eco_i, Flag=SF, ConnectionCount=60.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
# 7. Convert and Export to GGUF for Ollama (Optional)
# !git clone https://github.com/ggerganov/llama.cpp.git
# !python llama.cpp/convert_hf_to_gguf.py ./aegis-defense-lora --outfile aegis-defense-q4.gguf
# print("Download aegis-defense-q4.gguf and run in local Ollama:")
# print("ollama create aegis-defense -f Modelfile")